# M1: static lead-specific Gaussian copula

M1 asks whether historical residual ranks show stable within-lead spatial association. It estimates one full-group correlation matrix from training pseudo-scores for each lead, then uses that matrix for every future origin at the same lead.

$$\widehat R_{g,\tau}=\operatorname{Corr}\left(\{\mathbf z_{g,\tau}^{(i)}:i\in\mathcal D_{\rm train},V_{g,\tau}^{(i)}=1\}\right).$$

The production estimator uses Ledoit--Wolf shrinkage and positive-definite stabilization. It is a dependence model, not a re-estimation of the frozen Chronos marginals.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file())
sys.path.insert(0, str(PROJECT_ROOT / 'src')) if str(PROJECT_ROOT / 'src') not in sys.path else None
from simcast.config import SimcastConfig, deep_merge, load_config
from simcast.fm.cache import load_pit_library
from simcast.dependence import StaticGaussianCopula
from simcast.cli.train_dependence import train_from_config
from simcast.cli.evaluate import evaluate_from_config

CONFIG_FILE = 'configs/powertech2027/transformer.yaml'
OVERRIDES = ()
FIT_IN_MEMORY = False     # Set True to estimate M1 without writing an artifact.
TRAIN_IF_MISSING = False
RUN_EVALUATION = False
OUTPUT_DIR = PROJECT_ROOT / 'runs' / 'notebook_walkthrough' / 'm1_static'
base = load_config(PROJECT_ROOT / CONFIG_FILE, overrides=OVERRIDES)
config = SimcastConfig.model_validate(deep_merge(base.model_dump(mode='python'), {'dependence': {'method': 'static_gaussian'}}))
CACHE_DIR = PROJECT_ROOT / config.output.cache_dir / (config.output.cache_name or f'liander2024_{config.data.entity_type}')
library = load_pit_library(CACHE_DIR, access='training'); ds = library.dataset
entity_ids = [str(x) for x in ds.entity_id.values]; K_g = len(entity_ids)
assert entity_ids == config.protocol.ordered_entity_ids and K_g == config.protocol.entity_count
print(f'group={config.data.entity_type}, K_g={K_g}')

## Training-only estimator

For each $\tau$, only complete training vectors enter the estimate. If one entity is unavailable, that $(i,\tau)$ vector contributes to no covariance entry. The resulting covariance is converted to a correlation matrix so that $R_{kk}=1$ and $R$ is symmetric positive definite after stabilization. The static sensitivity `share_across_leads: true` instead pools leads; the primary M1 keeps leads separate.

In [ ]:
if FIT_IN_MEMORY:
    train = np.asarray(ds['split'].values) == 'train'
    z_train = np.asarray(ds['pit_z'].values)[train]
    model = StaticGaussianCopula(shrinkage=config.dependence.static_gaussian.shrinkage, share_across_leads=False, jitter=config.dependence.static_gaussian.jitter).fit(z_train, entity_ids)
    lead = 1; R = model.correlation_matrix(lead=lead)
    assert R.shape == (K_g, K_g)
    fig, ax = plt.subplots(figsize=(6, 5))
    image = ax.imshow(R, vmin=-1, vmax=1, cmap='coolwarm')
    ax.set(title=f'M1 training correlation, lead {lead}', xlabel='entity index', ylabel='entity index')
    fig.colorbar(image, ax=ax, label='correlation'); plt.show()
else:
    print('No fitting in default read-only mode; set FIT_IN_MEMORY=True to display a training-only M1 estimate.')

## Scientific interpretation

Large off-diagonal entries indicate persistent association of finite-PIT residual ranks at that lead. They do not identify a physical transmission mechanism and do not imply the correlation is unchanged at every origin. M1 is the essential reference for asking whether conditional methods extract useful *time-varying* dependence beyond a well-estimated static structure.

In [ ]:
run_dir = OUTPUT_DIR / 'static_gaussian'
if TRAIN_IF_MISSING and not run_dir.exists():
    run_dir = train_from_config(config, cache_dir=CACHE_DIR, output_dir=run_dir)
if RUN_EVALUATION:
    if not run_dir.exists(): raise FileNotFoundError('Set TRAIN_IF_MISSING=True or choose an existing M1 run.')
    evaluate_from_config(config, methods=('static_gaussian',), method_runs={'static_gaussian': run_dir}, cache_dir=CACHE_DIR, output_dir=OUTPUT_DIR / 'evaluation')
else:
    print('Read-only mode: no model fitting, run, or evaluation artifact is written.')

In [ ]:
import pandas as pd
from IPython.display import display
metrics_file = OUTPUT_DIR / 'evaluation' / 'metrics_by_lead.csv'
if metrics_file.is_file():
    metrics = pd.read_csv(metrics_file)
    display(metrics.groupby('method', as_index=False).mean(numeric_only=True))
    metrics.pivot(index='lead', columns='method', values='mean_pinball').plot(title='M1 aggregate pinball loss by lead')
else:
    print('No notebook evaluation table yet; existing CLI results can be copied by setting OUTPUT_DIR to their evaluation parent.')